# Analyse de la taille des AOs  Combien de chars ?

Objectif : mesurer le nombre de caractères, mots et tokens estimés des vrais documents AO (RC + CPS)
pour calibrer la limite `AO_MAX_CHARS` dans la config.

**Documents analysés :** RC (Règlement de Consultation) et CPS (Cahier des Prescriptions Spéciales)
 les deux fichiers clés qu'un soumissionnaire reçoit et doit lire pour répondre à un AO.

**Deux types de PDFs testés :**
- **PDF natif** : texte embarqué → extraction directe via `pymupdf`
- **PDF scanné** : image → OCR via `gpt-4o-mini` (modèle vision le moins cher)

**Seuil de détection scan** : si pymupdf extrait moins de 100 chars par page → considéré scanné.

In [4]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv

import fitz          # pymupdf
import pandas as pd
from openai import OpenAI

load_dotenv(dotenv_path=Path("../../.env"))

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Seuil : si moins de N chars par page → PDF scanné
SCAN_THRESHOLD_PER_PAGE = 100
# Modèle OCR le moins cher avec vision
OCR_MODEL = "gpt-4o-mini"
# Dossier contenant les exemples d'AO réels
AO_DIR = Path(r"C:\Users\eeljazouli\Documents\perso\swisstransfer_ab_consulting\Exemples d'AO")

print(f"OpenAI key chargée : {'oui' if os.getenv('OPENAI_API_KEY') else 'NON'}")
print(f"Dossier AO : {AO_DIR}")
print(f"Dossier existe : {AO_DIR.exists()}")

OpenAI key chargée : oui
Dossier AO : C:\Users\eeljazouli\Documents\perso\swisstransfer_ab_consulting\Exemples d'AO
Dossier existe : True


In [8]:
# ── Liste des fichiers RC et CPS uniquement ────────────────────────────────
# On prend les fichiers dont le nom commence par RC ou CPS (insensible à la casse)
# et on déduplique : si le même fichier existe dans 0-AO/ et 1-Dossier/, on garde un seul.

all_pdfs = sorted(AO_DIR.rglob("*.pdf"))

# Filtrer : nom commence par RC ou CPS
rc_cps = [
    p for p in all_pdfs
    if p.name.upper().startswith(("CPS"))
]

# Dédupliquer par nom normalisé (garder la première occurrence = dossier 0-)
seen = set()
pdfs_uniques = []
for p in rc_cps:
    key = p.name.upper().replace(" ", "").replace("-", "").replace("_", "")
    if key not in seen:
        seen.add(key)
        pdfs_uniques.append(p)

print(f"{len(rc_cps)} fichiers CPS trouvés, {len(pdfs_uniques)} après déduplication.\n")
for p in pdfs_uniques:
    print(f"  [{p.parent.parent.name}] {p.name}")

7 fichiers CPS trouvés, 7 après déduplication.

  [DAO 18_ANEF-Formation] CPS LOT 01 AO 18.pdf
  [DAO 18_ANEF-Formation] CPS LOT 2 AO N°18.pdf
  [DAO 18_ANEF-Formation] CPS LOT 3 AO N° 18.pdf
  [DAO 27-2025-ONSSA MASA] CPS 27-2025.pdf
  [DAO N¯02-2025-DRCADT] CPS AO N¯02-2025-DRCADT.pdf
  [DAO N° 61-2025-Evaluation FIDA DPA TAZA] CPS AO N¯ 61-2025.pdf
  [DAOO 13-2025-CRISM] CPS AOO 13-2025-CRISM.PDF


In [7]:
# ── Fonctions d'extraction ────────────────────────────────────────────────

def extract_text_pymupdf(pdf_path: Path) -> tuple[str, int, bool]:
    """Extrait le texte d'un PDF via pymupdf. Retourne (texte, nb_pages, est_scanné)."""
    doc = fitz.open(str(pdf_path))
    pages_text = [page.get_text() for page in doc]
    nb_pages = len(pages_text)
    full_text = "\n".join(pages_text)
    chars_per_page = len(full_text) / nb_pages if nb_pages else 0
    est_scanne = chars_per_page < SCAN_THRESHOLD_PER_PAGE
    doc.close()
    return full_text, nb_pages, est_scanne


def ocr_page_with_llm(pdf_path: Path, page_num: int) -> str:
    """OCR d'une page via gpt-4o-mini vision."""
    doc = fitz.open(str(pdf_path))
    page = doc[page_num]
    pix = page.get_pixmap(dpi=150)
    img_bytes = pix.tobytes("png")
    doc.close()
    b64 = base64.b64encode(img_bytes).decode()

    response = client.chat.completions.create(
        model=OCR_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a document transcription assistant. Transcribe all text visible in the image exactly as it appears. Return only the transcribed text, no commentary."
            },
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": "Transcris tout le texte visible dans ce document."}
                ]
            }
        ],
        max_tokens=2000,
        temperature=0,
    )
    return response.choices[0].message.content or ""


def extract_text(pdf_path: Path, ocr_max_pages: int = 5) -> dict:
    """Pipeline complet : pymupdf d'abord, OCR si scanné."""
    texte, nb_pages, est_scanne = extract_text_pymupdf(pdf_path)

    if est_scanne:
        pages_a_ocr = min(nb_pages, ocr_max_pages)
        texte_parts = [ocr_page_with_llm(pdf_path, i) for i in range(pages_a_ocr)]
        texte = "\n".join(texte_parts)
        if pages_a_ocr < nb_pages:
            texte += f"\n[... {nb_pages - pages_a_ocr} pages supplémentaires non OCRisées]"

    nb_chars = len(texte)
    nb_mots  = len(texte.split())
    tokens_estimes = nb_chars // 4  # ~1 token / 4 chars (approximation française)

    return {
        "fichier":        pdf_path.name,
        "ao":             pdf_path.parent.parent.name,
        "type":           "RC" if pdf_path.name.upper().startswith("RC") else "CPS",
        "pdf_type":       "scanné" if est_scanne else "natif",
        "nb_pages":       nb_pages,
        "nb_chars":       nb_chars,
        "nb_mots":        nb_mots,
        "tokens_estimes": tokens_estimes,
        "% couvert 2000": f"{min(nb_chars, 2000) / nb_chars * 100:.0f}%" if nb_chars else "0%",
        "% couvert 6000": f"{min(nb_chars, 6000) / nb_chars * 100:.0f}%" if nb_chars else "0%",
        "texte_extrait":  texte,
    }

print("Fonctions définies.")

Fonctions définies.


In [ ]:
# ── Extraction de tous les RC/CPS ─────────────────────────────────────────

results = []
for pdf in [pdfs_uniques[3]]:
    print(f"Traitement : [{pdf.parent.parent.name}] {pdf.name} ...", end=" ")
    try:
        r = extract_text(pdf)
        results.append(r)
        print(f"[{r['pdf_type']}] {r['nb_chars']:,} chars / {r['nb_mots']:,} mots / ~{r['tokens_estimes']:,} tokens")
    except Exception as e:
        print(f"ERREUR : {e}")

print(f"\n{len(results)}/{len(pdfs_uniques)} documents traités.")

Traitement : [DAO 27-2025-ONSSA MASA] CPS 27-2025.pdf ... [natif] 42,235 chars / 6,576 mots / ~10,558 tokens

1/7 documents traités.


In [20]:
r["texte_extrait"]

'Royaume du Maroc\nOffice National de sécurité Sanitaire\ndes Produits Alimentaires\nNO D\'APPEL D\'OFFRES\nCAH I E R\';;-ilil;;;;il;;;Ec IAL ES\nAppel d\'Offres ouvert national\nMASA\n;i*È1ll:id"lâ{l\nË*r-cJ, & lr.sl. (,r.;ûl,trl "...\'.irll\nii.:làèJl ;l^\'::"ll\nsur offres de prix No 27ty202stoNSSA/DR\nAssistance et accompagnement pour\nphytosanitaire pour le compte de la\nMarrakech Safi, lot unique.\nla mise à niveau des pépinières sur le plan\ndirection régionale de I,ONSSA, région de\nOBJET\nMODE DE PASSATION\nADRESSE\nTELEPHONE\nFAX\nMarché passé par appel d\'offres ouvert national sur offres de prix en séance\npublique en application des dispositions du décret N. 2-22-431 relatif aux marches\npublics du l5 chaabane 1444 (g MARS 2023) relatifaux marchés publics\nBoulevard Hassan II, premier étage, immeuble de la Direction Régionale de\nl\'agriculture, Gueliz - Marrakech\n+212 5 24 42 00 73 I +212 5 24 42 33 83\n+212 05 24 42 35 26\n#"k\'\n)doNss,r\n\nTable des matières\nPREAMBUL

In [16]:
pdfs_uniques[3]

WindowsPath("C:/Users/eeljazouli/Documents/perso/swisstransfer_ab_consulting/Exemples d'AO/DAO 27-2025-ONSSA MASA/0-DAO 27-2025-ONSSA MASA/CPS 27-2025.pdf")

In [14]:
# ── Tableau de résultats ───────────────────────────────────────────────────

df = pd.DataFrame(results).drop(columns=["texte_extrait"])
df = df.sort_values("nb_chars", ascending=False)
print(df.to_string(index=False))

        fichier                     ao type pdf_type  nb_pages  nb_chars  nb_mots  tokens_estimes % couvert 2000 % couvert 6000
CPS 27-2025.pdf DAO 27-2025-ONSSA MASA  CPS    natif        18     42235     6576           10558             5%            14%


In [15]:
# ── Statistiques et recommandation AO_MAX_CHARS ───────────────────────────

df_num = pd.DataFrame(results).drop(columns=["texte_extrait"])

print("=== Statistiques par type (RC vs CPS) ===")
print(df_num.groupby("type")[["nb_chars", "nb_mots", "tokens_estimes"]].describe().round(0).astype(int))

print("\n=== Statistiques globales ===")
print(df_num[["nb_chars", "nb_mots", "tokens_estimes"]].describe().round(0).astype(int))

print("\n=== Recommandation AO_MAX_CHARS ===")
for pct in [50, 75, 90, 100]:
    val = int(df_num["nb_chars"].quantile(pct / 100))
    print(f"  {pct:3d}e percentile : {val:>8,} chars  (~{val//4:,} tokens)")

print(f"\n  → Valeur actuelle dans config : 6 000 chars (~1 500 tokens)")
print(f"  → PDFs natifs  : {(df_num['pdf_type']=='natif').sum()}")
print(f"  → PDFs scannés : {(df_num['pdf_type']=='scanné').sum()}")

=== Statistiques par type (RC vs CPS) ===


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')

In [ ]:
# ── Aperçu texte extrait (debug) ──────────────────────────────────────────
# Modifier NOM_FICHIER pour inspecter un document

NOM_FICHIER = "CPS 27-2025.pdf"   # ← changer ici
DEBUT   = 0                        # offset de départ
LONGUEUR = 3000                    # nb de chars à afficher

doc = next((r for r in results if r["fichier"] == NOM_FICHIER), None)
if doc:
    texte = doc["texte_extrait"]
    print(f"Fichier : {doc['fichier']} ({doc['pdf_type']}, {doc['nb_chars']:,} chars, {doc['nb_pages']} pages)")
    print(f"Affichage chars [{DEBUT}:{DEBUT+LONGUEUR}]")
    print("-" * 60)
    print(texte[DEBUT:DEBUT + LONGUEUR])
    if len(texte) > DEBUT + LONGUEUR:
        print(f"\n[... {len(texte) - DEBUT - LONGUEUR:,} chars supplémentaires non affichés]")
else:
    print(f"Fichier '{NOM_FICHIER}' non trouvé.")
    print(f"Disponibles : {[r['fichier'] for r in results]}")